# Rule Explanation Demo (Non-Technical Overview)

This notebook demonstrates how business rules written in MVEL (a technical language) can be automatically translated into clear, plain English requirements. The goal is to make complex rule logic understandable for everyone, regardless of technical background.

**What you'll see:**
- How a rule is explained in simple English
- How the system checks if the explanation matches business expectations
- How English requirements can be converted back into rule logic

---


In [2]:
import os
import re
import sys
from pathlib import Path
from typing import List, Dict
from langchain_core.prompts import ChatPromptTemplate
import re
import numpy as np
import pandas as pd
import accelerate
import requests
import rapidfuzz
import json
from langchain_core.output_parsers import StrOutputParser
sys.path.append(str(Path.cwd().parent))  # Add parent directory to sys.path for local imports

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from agent.agents.explainer import explain_rule
from agent.tools.mvel_parser_tool import parse_mvel_branches
from agent.llm import get_llm

from sentence_transformers import SentenceTransformer, util

c:\Users\fahud\rule_config\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Appendix: Technical Details

The following cells contain the technical code and helper functions used by the notebook. Non-technical users can ignore this section.


In [3]:
# EXPLAIN_PROMPT: Converts a parsed rule structure (JSON) into clear, non-technical English with a summary and bullet points. Used for the main explanation pipeline.
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You write for non-technical stakeholders. "
        "Do not mention code, syntax, or programming terms."
    ),
    (
        "user",
        "Convert the following extracted rule structure into clear English.\n\n"
        "Requirements:\n"
        "- Start with a 1–2 sentence Summary\n"
        "- Then list Decision logic as bullet points\n"
        "- One bullet per branch, in order\n"
        "- Use 'Otherwise,' for the DEFAULT branch\n"
        "- Use provided context to define business terms if relevant\n\n"
        "Context:\n"
        "{context}\n\n"     
        "Rule extraction (JSON):\n"
        "{extraction_json}"
    )
])

# EXPLAIN_V2: Converts raw MVEL text into plain English step-by-step, without parsing. Used as a fallback when parsing fails.
EXPLAIN_V2 = ChatPromptTemplate.from_messages([
    (
        "system",
        "You write for non-technical stakeholders. "
        "Do not mention code, syntax, or programming terms."
    ),
    (
        "user",
        "Convert the following MVEL text into clear, natural English.\n"
        "Explain what the expression does in plain language.\n"
        "Break down each part of the expression step by step.\n"
        "If there are conditions, operators, or functions, describe what they mean.\n"
        "Write the description in sentences. \n"
        "Provide the final meaning as if explaining to someone with no programming background.\n"
        "MVEL TEXT:\n"
        "{mvel_text}"
    )
])


# ENGLISH_TO_MVEL: Converts a plain English requirement into a valid MVEL expression. Ensures the output is production-ready and precise.
ENGLISH_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n"
        "ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n(input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')\nUse grouped conditions with parentheses and logical operators (&&, ||) exactly in this style."
        "{english_text}"
    )
])

from langchain_core.prompts import ChatPromptTemplate

# ONE_SHOT_E_TO_MVEL: One-shot prompt for converting English requirements to MVEL, with a single example for guidance.
ONE_SHOT_E_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n"
        "EXAMPLE:\n"
        "ENGLISH REQUIREMENT:\n"
        "Message must be 'MT', estimate must be 'Actual', and clientId must be null or equal to 'N'.\n\n"
        "MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "NOW CONVERT THE FOLLOWING\nd"
        "ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n(input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')\nUse grouped conditions with parentheses and logical operators (&&, ||) exactly in this style."
        "{english_text}"
    )
])

# MULTI_SHOT_E_TO_MVEL: Multi-shot prompt for converting English requirements to MVEL, providing several examples for better accuracy.
MULTI_SHOT_E_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n\n"
        "EXAMPLE:\n\n"
        "FIRST ENGLISH REQUIREMENT:\n"
        "Message must be 'MT', estimate must be 'Actual', and clientId must be null or equal to 'N'.\n\n"
        "FIRST MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "SECOND ENGLISH REQUIREMENT:\n"
        "This expression returns true only if message equals 'CASH' and estimateCode equals 'Actual', "
        "ignoring case and ensuring neither value is null.\n\n"
        "SECOND MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('CASH')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual'))\n\n"
        "THIRD ENGLISH REQUIREMENT:\n"
        "This expression returns true only if message equals 'stoink', estimateCode equals 'Actual', "
        "and sender equals 'STOINK ULTRA PRO MAX', ignoring case and ensuring none of the values are null.\n\n"
        "THIRD MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('stoink')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual')) "
        "&& (input.sender != null && input.sender.equalsIgnoreCase('STOINK ULTRA PRO MAX'))\n\n"
        "NOW CONVERT THE FOLLOWING ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n"
        "Use grouped conditions with parentheses and logical operators (&&, ||) exactly in this style.\n\n"
        "ENGLISH REQUIREMENT:\n"
        "{english_text}"
    )
])

# ONE_SHOT_MVEL_TO_E: One-shot prompt for converting a MVEL expression to a single-sentence English requirement, with explicit translation rules and an example.
ONE_SHOT_MVEL_TO_E = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in interpreting MVEL rules used in business systems. "
        "Convert MVEL boolean expressions into clear, concise English requirements."
    ),
    (
        "user",
        "Convert the following MVEL expression into a clear English requirement.\n"
        "Translate the logic precisely.\n"
        "Explain null-check logic explicitly (e.g., 'is null' / 'is not null').\n"
        "Interpret equalsIgnoreCase as 'equals (case-insensitive)'.\n"
        "Preserve AND/OR relationships exactly.\n"
        "Return ONLY the English requirement as a single sentence.\n"
        "Do not include explanations.\n\n"
        "<instructions>\n"
        "1) Identify each top-level condition group separated by logical operators (&&, ||).\n"
        "2) For each group, translate null checks:\n"
        "   - \"x != null\" -> \"x is not null\"\n"
        "   - \"x == null\" -> \"x is null\"\n"
        "3) Translate string comparisons:\n"
        "   - \"x.equalsIgnoreCase('VALUE')\" -> \"x equals 'VALUE' (case-insensitive)\"\n"
        "   - \"x == 'VALUE'\" -> \"x equals 'VALUE'\"\n"
        "4) Preserve operator meaning and grouping exactly:\n"
        "   - \"&&\" -> \"and\"\n"
        "   - \"||\" -> \"or\"\n"
        "   - Parentheses indicate grouping and must be reflected in the English logic.\n"
        "5) Combine the translated groups into a single sentence that clearly states when the expression returns true.\n"
        "6) Return ONLY the final English requirement sentence. Do not include the steps or any extra text.\n\n"
        "<instructions>\n"
        "EXAMPLES:\n\n"
        "EXAMPLE:\n"
        "MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "ENGLISH REQUIREMENT:\n"
        "+stimate equals 'Actual' (case-insensitive), and clientId is either null or equals 'N'.\n\n"
        "NOW CONVERT THE FOLLOWING MVEL EXPRESSION:\n"
        "{mvel_text}"
    )
])

# MULTI_SHOT_MVEL_TO_E: Multi-shot prompt for converting MVEL expressions to English, with multiple examples for improved reliability.
MULTI_SHOT_MVEL_TO_E = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in interpreting MVEL rules used in business systems. "
        "Convert MVEL boolean expressions into clear, concise English requirements."
    ),
    (
        "user",
        "Convert the following MVEL expression into a clear English requirement.\n"
        "Translate the logic precisely.\n"
        "Explain null-check logic explicitly (e.g., 'is null' / 'is not null').\n"
        "Interpret equalsIgnoreCase as 'equals (case-insensitive)'.\n"
        "Preserve AND/OR relationships exactly.\n"
        "Return ONLY the English requirement as a single sentence.\n"
        "Do not include explanations.\n\n"
        "<instructions>\n"
        "1) Identify each top-level condition group separated by logical operators (&&, ||).\n"
        "2) For each group, translate null checks:\n"
        "   - \"x != null\" -> \"x is not null\"\n"
        "   - \"x == null\" -> \"x is null\"\n"
        "3) Translate string comparisons:\n"
        "   - \"x.equalsIgnoreCase('VALUE')\" -> \"x equals 'VALUE' (case-insensitive)\"\n"
        "   - \"x == 'VALUE'\" -> \"x equals 'VALUE'\"\n"
        "4) Preserve operator meaning and grouping exactly:\n"
        "   - \"&&\" -> \"and\"\n"
        "   - \"||\" -> \"or\"\n"
        "   - Parentheses indicate grouping and must be reflected in the English logic.\n"
        "5) Combine the translated groups into a single sentence that clearly states when the expression returns true.\n"
        "6) Return ONLY the final English requirement sentence. Do not include the steps or any extra text.\n\n"
        "<instructions>\n"
        "EXAMPLES:\n\n"
        "EXAMPLES:\n\n"
        "FIRST MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "FIRST ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'MT' (case-insensitive) and estimate equals 'Actual' (case-insensitive), and clientId is either null or equals 'N'.\n\n"
        "SECOND MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('CASH')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual'))\n\n"
        "SECOND ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'CASH' (case-insensitive) and estimateCode equals 'Actual' (case-insensitive), and neither value is null.\n\n"
        "THIRD MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('stoink')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual')) "
        "&& (input.sender != null && input.sender.equalsIgnoreCase('STOINK ULTRA PRO MAX'))\n\n"
        "THIRD ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'stoink' (case-insensitive), estimateCode equals 'Actual' (case-insensitive), and sender equals 'STOINK ULTRA PRO MAX' (case-insensitive), and none of those values are null.\n\n"
        "NOW CONVERT THE FOLLOWING MVEL EXPRESSION:\n"
        "{mvel_text}"
    )
])

# JUDGE_PROMPT: Compares two English explanations and selects the best one based on clarity, correctness, and structure for non-technical readers.
JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an impartial judge evaluating two explanations written for non-technical stakeholders.\n"
        "You must return ONLY valid JSON with no extra text."
    ),
    (
        "user",
        "Evaluate which explanation better satisfies the requirements below.\n\n"
        "Evaluation Criteria (in priority order):\n"
        "1) Clarity for non-technical stakeholders\n"
        "2) Correct interpretation of the rule logic\n"
        "3) Proper structure:\n"
        "   - Starts with a 1–2 sentence Summary\n"
        "   - Then lists Decision logic as bullet points\n"
        "   - One bullet per branch, in order\n"
        "   - Uses 'Otherwise,' for the DEFAULT branch\n"
        "4) Uses provided context appropriately to define business terms\n"
        "5) Does NOT mention code, syntax, JSON, or programming terms\n\n"
        "Rules:\n"
        "- Prefer explanations that are accurate and easy to understand.\n"
        "- Penalize any mention of technical or programming language.\n"
        "- If both are flawed, choose the less flawed one.\n"
        "- Ignore formatting differences unless they affect clarity or correctness.\n"
        "- Do NOT be influenced by length alone.\n\n"
        "Context:\n"
        "{context}\n\n"
        "Rule extraction (JSON):\n"
        "{extraction_json}\n\n"
        "EXPLANATION A:\n"
        "{response_a}\n\n"
        "EXPLANATION B:\n"
        "{response_b}\n\n"
        "Return ONLY valid JSON:\n"
        "{{\n"
        '  "winner": "A" | "B",\n'
        '  "confidence": 0.0-1.0,\n'
        '  "reason": "1-3 sentences explaining the key deciding factors"\n'
        "}}"
    ),
])


In [4]:
# Initialize an LLM and provide well-named helper functions for the notebook
# Keeps functions explicit and easy for users to call.
llm = get_llm(model="llama3.1", temperature=0.0)


def explain_parsed_rule(rule_text: str, context: str = "Business context or glossary here") -> str:
    """
    Parse MVEL text, run the explain prompt pipeline, and return English.
    Uses the parsed extraction JSON as the prompt input.
    Preferred method for generating clear explanations from structured rules.
    """
    parsed = parse_mvel_branches(rule_text)
    extraction_json = json.dumps(parsed, ensure_ascii=False, indent=2)
    chain = EXPLAIN_PROMPT | llm | StrOutputParser()
    print("[explain_parsed_rule] Running explanation chain on parsed extraction...")
    explanation = chain.invoke({"extraction_json": extraction_json, "context": context}).strip()
    print("[explain_parsed_rule] Completed.")
    return explanation


def explain_raw_mvel(rule_text: str) -> str:
    """
    Explain raw MVEL text without parsing. Useful when parser fails.
    Produces a step-by-step plain English explanation for any MVEL rule.
    """
    chain = EXPLAIN_V2 | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content


def explain_raw_mvel_one(rule_text: str) -> str:
    """
    Explain raw MVEL text using the one-shot prompt. Returns a concise English requirement sentence.
    """
    chain = ONE_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content


def explain_raw_mvel_multi(rule_text: str) -> str:
    """
    Explain raw MVEL text using the multi-shot prompt. Returns a concise English requirement sentence, leveraging multiple examples.
    """
    chain = MULTI_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content


def convert_english_to_mvel(english_text: str) -> str:
    """
    Convert a plain-English requirement into an MVEL expression via the LLM.
    Returns the raw string returned by the model (intended to be a single MVEL expression).
    Uses the main English-to-MVEL prompt.
    """
    chain = ENGLISH_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content


def convert_english_to_mvel_one(english_text: str) -> str:
    """
    Convert a plain-English requirement into an MVEL expression using the one-shot prompt.
    Returns a single MVEL expression string.
    """
    chain = ONE_SHOT_E_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content


def convert_english_to_mvel_multi(english_text: str) -> str:
    """
    Convert a plain-English requirement into an MVEL expression using the multi-shot prompt.
    Returns a single MVEL expression string, leveraging multiple examples for accuracy.
    """
    chain = MULTI_SHOT_E_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content


def chain_of_thought(rule_text):
    """
    Runs the one-shot MVEL-to-English prompt multiple times for self-consistency checking.
    Returns the raw English explanations for further analysis.
    """
    chain = ONE_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    return response.content
    

In [10]:
# Utilities for fuzzy matching and evaluation against a CSV library of rules
import re
import pandas as pd
from rapidfuzz import fuzz
model = SentenceTransformer("all-MiniLM-L6-v2")
CSV_PATH = "pre_match.csv"

# Example test MVELs (replace or extend as needed)
tests = [
    "(input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')",
    "(input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType.equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') && (input.transactionIndex == 'N') && (input.Ref.contains('CASH MONEY HEROES'))",
    "(input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transactionCode.equalsIgnoreCase('CAPITAL HILL')) && (input.entityFlag != null && input.entityFlag == 'Y')",
]

english_texts = []  # populated by evaluate


def normalize(text: str) -> str:
    """
    Cleans and normalizes a string for comparison (removes extra spaces, fixes logical operator typos).
    """
    t = str(text).strip()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" | | ", " || ")
    return t


def similarity(a: str, b: str) -> float:
    """
    Returns a similarity score (0..1) between two strings using rapidfuzz token_set_ratio.
    Used for fuzzy matching of rules and explanations.
    """
    return fuzz.token_set_ratio(normalize(a), normalize(b)) / 100.0


def cosine_sim(a, b):
    """
    Computes cosine similarity between two texts using sentence embeddings.
    Used for semantic comparison of explanations and targets.
    """
    embedding1 = model.encode(a, convert_to_tensor=True)
    embedding2 = model.encode(b, convert_to_tensor=True)
    similarity = util.cos_sim(embedding1, embedding2)
    return similarity.item()


def best_target_match(model_output: str, library_targets: list[str], similarity) -> tuple[int, float]:
    """
    Finds the best-matching target from a list, given a model output and a similarity function.
    Returns the index and score of the best match.
    """
    scores = []
    for t in library_targets:
        score = similarity(model_output, t)
        scores.append(score)
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    print("generated output:", model_output)
    print("matched label:", library_targets[best_idx])
    print(f"{similarity}", float(scores[best_idx]))
    return best_idx, float(scores[best_idx])


def evaluate(explain_fn):
    """
    Evaluate a given explanation function against the CSV library.
    explain_fn: callable that accepts an MVEL string and returns English text.
    Runs the explanation and compares it to reference targets.
    """
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["RULE", "TARGET"]).reset_index(drop=True)

    library_rules = df["RULE"].astype(str).tolist()
    library_targets = df["TARGET"].astype(str).tolist()

    print(f"Loaded library rows: {len(library_rules)}")

    for i, test_rule in enumerate(tests, start=1):
        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("TEST RULE:\n", test_rule)

        # Call the provided explain function
        model_output = explain_fn(test_rule)
        english_texts.append(model_output)

        best_idx, best_score = best_target_match(model_output, library_targets, similarity=cosine_sim)


def eval_convert(conversion):
    """
    Converts generated English explanations back to MVEL using a conversion function.
    Compares the result to the original rule for accuracy.
    """
    df = pd.read_csv(CSV_PATH)
    library_rules = df["RULE"].astype(str).tolist()
    explanation = df["TARGET"].astype(str).tolist()

    for idx, rule in enumerate(english_texts):
        # Call conversion and normalize to a single-line MVEL string
        output = conversion(rule)
        output_str = str(output)
        clean_rule = output_str.replace('`', '').replace('\n', ' ').strip()

        # Find best match in the dataset (compare converted MVEL to library rules)
        best_idx, best_score = best_target_match(clean_rule, library_targets=library_rules, similarity=similarity)


def self_consistency_cot(runs: int = 3):
    """
    Runs the explanation process multiple times to check for consistency in outputs.
    Useful for evaluating the reliability of the explanation pipeline.
    """
    result = []
    df = pd.read_csv(CSV_PATH)
    library_targets = df["TARGET"].astype(str).tolist()
    for _ in range(runs):
        result.append(chain_of_thought(tests[0]))
    print(result)
    
    res_concat = ";".join(result)
    prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistance and answer precise and concise"),
    ("user",
     "You will get multiple answers in <<>>, separated by ; "
     "<<{res_concat}>> Extract the best answer among those answers.")
])
    chain = prompt | llm 
    response = chain.invoke({"res_concat": res_concat})
    output = str(response.content)
    clean_rule = output.replace('`', '').replace('\n', ' ').strip()
    best_idx, best_score = best_target_match(clean_rule, library_targets=library_targets, similarity=similarity)
    print(output)
        

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 824.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Run evaluation using the parsed-rule explainer (preferred when parser works)
evaluate(explain_fn=explain_parsed_rule)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_parsed_rule] Running explanation chain on parsed extraction...
[explain_parsed_rule] Completed.
generated output: Here is the extracted rule structure rewritten in clear English:

**Summary**
This rule checks if a message meets certain criteria and advises on whether to proceed with a transaction.

**Decision Logic**

* The message must be 'MT', 'saipan', or 'CA' (case-insensitive) and not null.
* The money code must be 'Actual' (case-insensitive) and not null.
* The transaction type must be 'CAPITAL HILL' (case-insensitive) and not null.
* Th

## 1. Explaining Rules in Plain English

Below, we take a technical business rule and automatically generate a clear English explanation. This helps non-technical stakeholders understand what the rule does, without needing to read code.

**Example:**
- The system will show the original rule and its English explanation.


In [12]:
evaluate(explain_fn=explain_raw_mvel)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.
generated output: Here's the explanation of the MVEL text in plain language:

**What the expression does:**
This expression checks if a set of conditions are met for an input message. If all conditions are true, then it returns true; otherwise, it returns false.

**Breaking down each part step by step:**

1. `input.message != null`: This condition checks if the "message" field in the input data is not empty or null.
2. `(input.message.equalsIgnoreCase('MT') || ... )`: If t

---

## 2. Alternative Explanation Methods

If the main parser cannot process a rule, we can still generate a plain English explanation using a backup method. This ensures every rule can be explained, even if it's written in an unusual way.


In [13]:
self_consistency_cot(3)

[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Running raw-text explanation...
["message equals 'MT' (case-insensitive) or message equals 'saipan' (case-insensitive) or message equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL', and advise equals 'Y'.", "message equals 'MT' (case-insensitive) or message equals 'saipan' (case-insensitive) or message equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL' (case-insensitive), and advise equals 'Y'.", "message equals 'MT' (case-insensitive) or message equals 'saipan' (case-insensitive) or message equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL' (case-insensitive), and advise equals 'Y'."]
generated output: The best answer is:  message equals 'MT' (case-insensitive) o

---

## 3. Consistency Check

We can run the explanation process several times to see if the results are consistent. This helps ensure the explanations are reliable and not random.


In [14]:
# After you have run `evaluate` to populate `english_texts`,
# you can attempt to convert the generated English back to MVEL and compare.
eval_convert(conversion=convert_english_to_mvel)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) &&  (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) &&  (input.transactionType != null && input.transactionType.equalsIgnoreCase('CAPITAL HILL')) &&  (input.adviseFlag != null && input.adviseFlag.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("nothing trade")) &&
(input.estimateCode != null && input.estimateCode.equalsIgnoreCase("Actual")) &&
(input.sender != null && input.sender.equalsIgnoreCase("DFS")) &&
(input.entityFlag != null && input.entityFlag.equals("N"))
<function similarity at 0x00000130887DBF60> 0.6746987951807228
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && i

---

## 4. Converting English Back to Rule Logic

After generating English explanations, we can try converting them back into technical rule logic. This checks if the explanations are precise enough to recreate the original rules.


In [15]:
eval_convert(conversion=convert_english_to_mvel_one)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) &&  (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) &&  (input.transactionType != null && input.transactionType.equalsIgnoreCase('CAPITAL HILL')) &&  (input.adviseFlag != null && input.adviseFlag.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("ABC")) &&
(input.moneyCode != null && input.moneyCode.equalsIgnoreCase("Money")) &&
(input.workType != null && input.workType.equalsIgnoreCase("regular")) &&
(input.transaction != null && input.transaction.equals("Y"))
<function similarity at 0x00000130887DBF60> 0.6597222222222223
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.

In [16]:
eval_convert(conversion=convert_english_to_mvel_multi)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))  && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual'))  && (input.transactionType != null && input.transactionType.equalsIgnoreCase('CAPITAL HILL'))  && (input.adviseFlag != null && input.adviseFlag.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("ABC")) &&
(input.moneyCode != null && input.moneyCode.equalsIgnoreCase("Money")) &&
(input.workType != null && input.workType.equalsIgnoreCase("regular")) &&
(input.transaction != null && input.transaction.equals("Y"))
<function similarity at 0x00000130887DBF60> 0.6597222222222223
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.

In [17]:
evaluate(explain_fn=explain_raw_mvel_one)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.
generated output: message equals 'MT' (case-insensitive) or message equals 'saipan' (case-insensitive) or message equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL' (case-insensitive), and advise equals 'Y'.
matched label: If the message is not null and equals "ABC"  where it is case-sensitive, and the moneyCode is not null and equals "Money"  where it is case-sensitive, and the workType is not null and e

In [18]:
evaluate(explain_fn=explain_raw_mvel_multi)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.
generated output: Return true only if message equals 'MT' (case-insensitive) or equals 'saipan' (case-insensitive) or equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL' (case-insensitive), and advise equals 'Y', and none of those values are null.
matched label: If the message is not null and equals "ABC"  where it is case-sensitive, and the moneyCode is not null and equals "Money"  where it is case-sensit

In [19]:
# step1_make_prompts.py
import json

IN_PATH = "pre_match_sft_chat.jsonl"
OUT_PATH = "prompts.jsonl"

def build_prompt(messages):
    prompt = ""
    for m in messages:
        if m["role"] in ["system", "user"]:
            prompt += f"{m['role'].upper()}:\n{m['content']}\n\n"
    prompt += "ASSISTANT:\n"
    return prompt

with open(IN_PATH, "r", encoding="utf-8") as f_in, \
     open(OUT_PATH, "w", encoding="utf-8") as f_out:

    for line in f_in:
        obj = json.loads(line)
        prompt = build_prompt(obj["messages"])
        f_out.write(json.dumps({"prompt": prompt}) + "\n")

print("prompts.jsonl created")


prompts.jsonl created


---

## 5. How Prompts and Candidates Are Built (For Reference)

The following sections show how the system builds prompts and generates possible explanations. These are included for transparency, but you can skip them if you only want to see the main results.


In [20]:
# step2_generate_candidates.py
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import accelerate

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # change if needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to("cpu")


with open("prompts.jsonl", "r") as f_in, \
     open("candidates.jsonl", "w") as f_out:

    for line in f_in:
        row = json.loads(line)
        prompt = row["prompt"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=4,
        )

        candidates = [
            tokenizer.decode(o, skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
            for o in outputs
        ]

        f_out.write(json.dumps({
            "prompt": prompt,
            "candidates": candidates
        }) + "\n")

print("candidates.jsonl created")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 453.96it/s, Materializing param=model.norm.weight]                              


candidates.jsonl created


In [21]:
import json
import random

# Reuse the llm initialized earlier in the notebook (via get_llm).
# If llm isn't defined yet, create it using get_llm.
try:
    llm  # existing llm from previous cells
except NameError:
    llm = get_llm(model="phi3:3.8b", temperature=0.0)

# judge: Compares two candidate explanations using the judge prompt and returns the winner ("A" or "B").
judge_chain = JUDGE_PROMPT | llm


def judge(a, b, prompt_blob):
    """
    Compares two candidate explanations using the judge prompt and returns the winner ("A" or "B").
    """
    result = judge_chain.invoke({
        "context": "",                  # keep empty
        "extraction_json": "",          # keep empty
        "response_a": a,
        "response_b": b,
    })
    return json.loads(result.content)["winner"]


def anchor_pick(candidates, prompt_blob):
    """
    Runs a mini-tournament among four candidate explanations to pick the best and worst, reducing bias.
    Returns the indices of the best and worst candidates.
    """
    wins = [0] * 4
    anchor = random.randrange(4)

    for i in range(4):
        if i == anchor:
            continue

        # flip A/B randomly to reduce bias
        if random.random() < 0.5:
            winner = judge(candidates[anchor], candidates[i], prompt_blob)
            if winner == "A":
                wins[anchor] += 1
            else:
                wins[i] += 1
        else:
            winner = judge(candidates[i], candidates[anchor], prompt_blob)
            if winner == "A":
                wins[i] += 1
            else:
                wins[anchor] += 1

    best = wins.index(max(wins))
    worst = wins.index(min(wins))
    return best, worst


# The following block runs the anchor_pick tournament for each prompt/candidate set and writes the best/worst to file.
with open("candidates.jsonl") as f_in, open("dpo_pairs.jsonl", "w") as f_out:
    for line in f_in:
        row = json.loads(line)

        best, worst = anchor_pick(row["candidates"], row["prompt"])

        f_out.write(json.dumps({
            "prompt": row["prompt"],
            "chosen": row["candidates"][best],
            "rejected": row["candidates"][worst],
        }) + "\n")


---

## 6. Judging and Selecting the Best Explanations

This section shows how the system automatically compares different explanations and picks the best one, using clear criteria focused on clarity and correctness for non-technical readers.


In [22]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dpo_pairs.jsonl")["train"]


Generating train split: 14 examples [00:00, 235.35 examples/s]


---

## 7. Data Preparation for Training (Optional)

This section loads the data pairs for further training or analysis. Non-technical users can skip this part unless interested in how the data is structured for machine learning.


In [23]:
# Safe model-load test — run in its own new cell
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, traceback

model_name = "mistralai/Mistral-7B-v0.1"

try:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer OK")
except Exception as e:
    print("Tokenizer load failed:", repr(e))
    traceback.print_exc()

try:
    print("Attempting model load with low_cpu_mem_usage + device_map='auto'...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    print("Model loaded with device_map='auto' OK")
except Exception as e:
    print("device_map='auto' load failed:", repr(e))
    traceback.print_exc()
    try:
        print("Falling back to CPU-only load...")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map={"": "cpu"})
        print("Model loaded on CPU (fallback) OK")
    except Exception as e2:
        print("CPU fallback failed:", repr(e2))
        traceback.print_exc()

Loading tokenizer...
Tokenizer OK
Attempting model load with low_cpu_mem_usage + device_map='auto'...


Loading weights: 100%|██████████| 291/291 [00:18<00:00, 15.84it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk and cpu.


Model loaded with device_map='auto' OK


---

## 8. Model Loading (Technical Reference)

This final section is for technical users who want to see how the language model is loaded. Most users can skip this part.
